# Corruption-robust O2O comparison

This notebook discovers the newest matching comparison without hard-coded protocol/profile paths. It consumes canonical run manifests, reporting CSVs, and plots produced by the training code; it does not reimplement final-score aggregation.

In [ ]:
# Edit this cell, then Run All.
ENVIRONMENT = "halfcheetah"  # halfcheetah, hopper, walker2d
DATASET = "medium-replay-v2"
CORRUPTION = "random"  # clean, random, adversarial
CORRUPTION_TARGET = "rewards"  # none, observations, actions, rewards, dynamics
TRAINING_STARTED_AT = None  # e.g. "2026-08-21 03:00:00"; None selects latest
RESULTS_ROOT = None  # None -> <repo>/results; or an absolute --output-root
INCLUDE_RUNNING = True  # True: live diagnostic view; False: completed valid suites only

In [ ]:
import json
import sys
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

ALIASES = {"half-cheetah": "halfcheetah", "walker": "walker2d"}
domain = ALIASES.get(ENVIRONMENT.lower(), ENVIRONMENT.lower())
env_name = f"{domain}-{DATASET}"
target = "none" if CORRUPTION == "clean" else CORRUPTION_TARGET

def project_root():
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (candidate / "robust_o2o").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate repository root")

def started_at(comparison_dir):
    manifest = comparison_dir / "manifest.json"
    if manifest.exists():
        value = json.loads(manifest.read_text(encoding="utf-8")).get("start_time")
        if value:
            return pd.Timestamp(value).to_pydatetime().replace(tzinfo=None)
    summaries = sorted((comparison_dir / "runs").rglob("summary.json"))
    starts = []
    for summary in summaries:
        value = json.loads(summary.read_text(encoding="utf-8")).get("start_time")
        if value:
            starts.append(pd.Timestamp(value).to_pydatetime().replace(tzinfo=None))
    return min(starts) if starts else datetime.fromtimestamp(comparison_dir.stat().st_mtime)

root = project_root()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
from plot_results import load_runs  # noqa: E402

results_root = Path(RESULTS_ROOT).expanduser().resolve() if RESULTS_ROOT else root / "results"
comparisons_root = results_root / "comparisons"
candidates = []
for runs_dir in comparisons_root.rglob("runs") if comparisons_root.exists() else []:
    configs = sorted(runs_dir.rglob("config.json"))
    if not configs:
        continue
    config = json.loads(configs[0].read_text(encoding="utf-8"))
    comparison_manifest_path = runs_dir.parent / "manifest.json"
    comparison_manifest = json.loads(comparison_manifest_path.read_text(encoding="utf-8")) if comparison_manifest_path.exists() else None
    if comparison_manifest and comparison_manifest.get("aggregation_error"):
        continue
    if not INCLUDE_RUNNING and (comparison_manifest is None or not comparison_manifest.get("benchmark_valid", False)):
        continue
    if INCLUDE_RUNNING and comparison_manifest is not None and not comparison_manifest.get("benchmark_valid", False):
        continue
    if (config.get("env_name"), config.get("corruption"), config.get("corruption_target")) == (env_name, CORRUPTION, target):
        candidates.append(runs_dir.parent)
if not candidates:
    raise FileNotFoundError(f"No comparison for {env_name} {CORRUPTION}/{target}")
if TRAINING_STARTED_AT is None:
    COMPARISON_DIR = max(candidates, key=started_at)
else:
    requested = pd.Timestamp(TRAINING_STARTED_AT).to_pydatetime().replace(tzinfo=None)
    exact = [path for path in candidates if started_at(path) == requested]
    if not exact:
        available = "\n".join(f"{started_at(path)}  {path}" for path in sorted(candidates, key=started_at))
        raise FileNotFoundError(f"No exact start time {requested}. Available:\n{available}")
    COMPARISON_DIR = exact[-1]
print(f"Comparison: {COMPARISON_DIR}")
COMPARISON_MANIFEST = json.loads((COMPARISON_DIR / "manifest.json").read_text(encoding="utf-8")) if (COMPARISON_DIR / "manifest.json").exists() else {}
if COMPARISON_MANIFEST.get("aggregation_error"):
    raise RuntimeError(f"Invalid comparison: {COMPARISON_MANIFEST['aggregation_error']}")
print(f"Started:    {started_at(COMPARISON_DIR)}")
print(f"Purpose:    {COMPARISON_MANIFEST.get('run_purpose', 'running')}")
print(f"Suite:      {COMPARISON_MANIFEST.get('suite_profile', 'running')}")
if INCLUDE_RUNNING:
    print("LIVE DIAGNOSTIC VIEW: running/partial curves are not final benchmark results.")

In [ ]:
runs = load_runs(COMPARISON_DIR / "runs")
if not INCLUDE_RUNNING:
    runs = runs[runs["run_status"] == "completed"]
if runs.empty:
    raise RuntimeError("No matching evaluation metrics")
identity_columns = ["algorithm", "seed", "implementation_fidelity", "condition_status", "run_purpose", "run_status"]
display(runs[identity_columns].drop_duplicates().sort_values(["algorithm", "seed"]))

for filename, label in [("paper_reproduction_summary.csv", "Paper-eligible source-primary reporting"), ("common_benchmark_summary.csv", "Common benchmark metric"), ("per_seed_final_scores.csv", "Publication-eligible per-seed source-primary scores")]:
    path = COMPARISON_DIR / filename
    print(f"\n{label}: {path.name}")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Not written yet (training may still be running).")

In [ ]:
for phase in ("offline_online", "offline", "online"):
    prefix = "diagnostic_running" if INCLUDE_RUNNING else "comparison"
    image_path = COMPARISON_DIR / f"{prefix}_{phase}.png"
    print(f"\n{phase.replace('_', ' → ')}")
    if image_path.exists():
        display(Image(filename=str(image_path)))
    else:
        print(f"Plot not available yet: {image_path}")